# Gold v2 — Q3 위치: expert 표시를 target 1장에 그려서 묻기

Vision Expert full에서는 heatmap·ROI를 **Image 2(별도 이미지)**로 줬고, 어떤 expert 조건도 Q3에서 원본만(image-only)을 넘지 못했습니다.
Oracle 실험에서는 위치 표시를 **target 위에 그린 1장**이 2장 방식보다 Q3에 훨씬 좋았습니다. 이 노트북은 그 1장 방식을 실제 expert 출력으로 시험합니다.

- 대상: Q3 42문항만 (Q1·Q2는 묻지 않음)
- 조건 (expert 4개 × 모델 4개)
  - `heatmap_single`: expert heatmap이 덧그려진 target 1장 (Vision Expert full의 pure heatmap overlay 그대로)
  - `roi_box_single`: target에 expert가 가장 의심한 영역을 빨간 사각형으로 그린 1장
- 비교: 원본만 / 같은 expert의 2장 조건(heatmap, roi) — 같은 문항 짝 비교(exact McNemar)
- 진단: `expert_only.csv` — LVLM 없이 expert ROI 중심을 3×3 영역으로 바꿔 답한 정확도. 이게 낮으면 형식이 아니라 expert 위치 자체가 문제

GPU: **L4** (4bit 금지). 호출 수 = 42 × expert 4 × 조건 2 × 모델 4 = 1,344회. 대략 30~50분.
끊기면 위에서부터 다시 실행하면 이어서 돕니다.

결과: `MyDrive/lvlm_insplad_results/gold_v2_q3_heatmap_single/` → GitHub `results/latest/gold_v2_q3_heatmap_single/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
VX_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_q3_heatmap_single"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
EXPERTS = ["patchcore","dinomaly","anomalyclip","reconpatch_lite"]
CONDITIONS = ["heatmap_single","roi_box_single"]
CHECKPOINT_EVERY = 20
PUBLISH = True

SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
import base64, csv, gc, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
VX_ROOT = DRIVE_ROOT / VX_FOLDER
VX_EVAL = VX_ROOT / "eval"
MERGED_ROOT = VX_ROOT / "merged_evidence"
MERGED_MANIFEST = MERGED_ROOT / "roi_manifest_all.csv"
VX_PRED_DIR = VX_EVAL / "predictions"
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    return subprocess.run(cmd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT":"1",
    "GIT_CONFIG_KEY_0":"http.extraHeader",
    "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT":"0",
})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env

# LVLM 추론만 하므로 anomalib 등 expert 패키지는 설치하지 않습니다.
run([sys.executable,"-m","pip","install","-q",
     "transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2",
     "pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))

import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음). 경로를 알면 설정 셀의 *_ZIP_PATH에 넣으세요.", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]


SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
print("SUP:", SUP_ZIP)
IMAGE_ONLY_CSV = REPO_DIR / "results/latest/gold_v2_baseline/prediction_review.csv"
need = [MERGED_MANIFEST, IMAGE_ONLY_CSV]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError(f"입력 없음: {missing}")
if not sorted(QA_DIR.glob("localization_qa_*.jsonl")):
    raise FileNotFoundError(f"localization_qa_*.jsonl under {QA_DIR}")
print("통합 manifest:", MERGED_MANIFEST)
print("2장 비교용 예측:", VX_PRED_DIR, "있음" if VX_PRED_DIR.is_dir() else "없음 (2장 비교 열은 비게 됨)")

In [ ]:
# 실행 — 끊기면 이 셀만 다시 실행해도 완료된 문항은 재사용
run([
    sys.executable, REPO_DIR / "scripts/run_gold_v2_q3_heatmap_single.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--roi-manifest", MERGED_MANIFEST,
    "--artifact-root", MERGED_ROOT,
    "--image-only-csv", IMAGE_ONLY_CSV,
    "--vx-pred-dir", VX_PRED_DIR,
    "--out-dir", RUN_ROOT,
    "--models", *MODELS,
    "--experts", *EXPERTS,
    "--conditions", *CONDITIONS,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
])

In [ ]:
# 결과
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
print("[expert만 (LVLM 없음)] ROI 중심 → 3x3 영역")
display(pd.read_csv(RUN_ROOT / "expert_only.csv"))
cols = [c for c in ["model_key","expert","condition","q3_acc","q3_acc_region_only","delta_vs_image_only",
                    "gain_vs_image_only","loss_vs_image_only","mcnemar_p_vs_image_only",
                    "two_image_q3_acc","delta_vs_two_image","mcnemar_p_vs_two_image","valid_output_rate"] if c in lb.columns]
display(lb[cols].round(3))
best = lb[lb.condition != "image_only"].sort_values("q3_acc", ascending=False).groupby("model_key").head(1)
print("\n[모델별 최고 조건 vs 원본만]  p < 0.05면 우연 아님 (42문항이라 큰 차이가 필요)")
display(best[cols].round(3))
print((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))

In [ ]:
# GitHub publish
if PUBLISH:
    publish_files = ["leaderboard.csv", "expert_only.csv", "run_status.json", "run_metadata.json"]
    run([
        sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
        "--out-dir", RUN_ROOT,
        "--repo", REPO,
        "--branch", REPO_REF,
        "--prefix", "gold_v2_q3_heatmap_single",
        "--files", *[f for f in publish_files if (RUN_ROOT / f).is_file()],
    ], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_q3_heatmap_single/")

## 완료 후

- `run_status.json`의 `row_errors`가 모두 0인지 확인 (0이 아니면 실행 셀만 다시 실행)
- 원본만 Q3: Qwen·MiniCPM 0.667, InternVL 0.548, LLaVA 0.524 — 이걸 넘는 조건이 있는지가 핵심
- `expert_only`가 낮으면(찍기 수준) expert 위치 자체가 문제 → 해상도·확대 쪽으로
- 42문항이라 p-value가 크게 나오기 쉬움. 방향이 모델·expert에 걸쳐 일관적인지도 같이 볼 것
- gold 42문항에서의 탐색 결과임 (run_metadata에 명시)